# INTELIGENCIA ARTIFICIAL

## Laboratorio 2: Búsqueda con Información (A* y heurísticas)
## Indicaciones generales:

- Las respuestas deben contar con **fundamento teórico**.
- Cualquier indicio de plagio resultará en la anulación de la prueba.
- Debe presentar sus respuestas en base a los resultados de ejecución en los casos que se solicite. **No se calificarán aquellas respuestas que no presenten un resultado de ejecución o que no concuerden con este**.
- Subir el cuadernillo con el nombre **Lab02_código.ipynb**, donde código es su código PUCP de 8 dígitos.
- Está prohibido el uso de herramientas de IA. En caso se detecte el uso de estas herramientas la calificación será 00.


**Usted deberá completar el código en las secciones indicadas con "COMPLETAR"**

# Busqueda en SpaceWorld

Un arqueólogo espacial ha recibido una misteriosa señal proveniente de un rincón remoto del universo. Ha conseguido encontrar su origen, pero necesita tu ayuda para que el piloto automático de su nave encuentre la mejor ruta para llegar hasta allí sin estrellarse contra el Sol.

El objetivo es llegar al origen de la señal (el objetivo 'E'). Además, el espacio tiene elementos, como  planetas y satélites, que el arqueologo le gustaria visitar en su camino ya que representa cierto "valor" al conocimiento cientifico. Sin embargo, su principal preocupación es el combustible de la nave, pues es limitado. Además, existen obstáculos, como asteroides, que requieren más combustible (costo) para atravesarlos, y el Sol, que es completamente impasable.

**Resumen de los elementos del espacio:**
- `'S'` (Sol): Impasable, costo infinito, valor 0.
- `'E'` (Objetivo): Costo 1, valor 100.
- `'P'` (Planeta): Costo -0.5 (puede recargar combustible en el planeta, por eso el costo es negativo), valor 15.
- `'T'` (Satélite): Costo -0.5 (puede recargar combustible en el satélite), valor 50.
- `'A'` (Asteroide): Costo 5, valor 0.
- `' '` (Espacio vacío): Costo 1, valor 0.

In [102]:
#Carga de librerías
import numpy as np
import random
import math
from collections import deque
import heapq
import plotly.graph_objects as go
from plotly.offline import iplot
import plotly.express as px
# Semilla
random.seed(42)
np.random.seed(42)

In [103]:
#@title Clases base - No modifique este código, solo ejecute la celda
class SearchProblem():
    def __init__(self, initial, goal=None):
        """Este constructor especifica el estado inicial y posiblemente el estado(s) objetivo(s),
        La subclase puede añadir mas argumentos."""
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        """Retorna las acciones que pueden ser ejecutadas en el estado dado.
        El resultado es tipicamente una lista."""
        raise NotImplementedError

    def result(self, state, action):
        """Retorna el estado que resulta de ejecutar la accion dada en el estado state.
        La accion debe ser alguna de self.actions(state)."""
        raise NotImplementedError

    def goal_test(self, state):
        """Retorna True si el estado pasado satisface el objetivo."""
        raise NotImplementedError

    def path_cost(self, c, state1, action, state2):
        """Retorna el costo del camino de state2 viniendo de state1 con
        la accion action, asumiendo un costo c para llegar hasta state1.
        El metodo por defecto cuesta 1 para cada paso en el camino."""
        return c + 1


class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        """Crea un nodo de arbol de busqueda, derivado del nodo parent y accion action"""
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.science_value = 0  # Valor científico acumulado
        self.depth = 0
        if parent:
            self.depth = parent.depth + 1

    def expand(self, problem):
        """Devuelve los nodos alcanzables en un paso a partir de este nodo."""
        return [self.child_node(problem, action)
                for action in problem.actions(self.state)]

    def child_node(self, problem, action):
        """Crea un nodo hijo para la acción dada"""
        next_state = problem.result(self.state, action)
        return Node(next_state, self, action,
                    problem.path_cost(self.path_cost, self.state, action, next_state))

    def solution(self):
        """Retorna la secuencia de acciones para ir de la raiz a este nodo."""
        return [node.action for node in self.path()[1:]]

    def path(self):
        """Retorna una lista de nodos formando un camino de la raiz a este nodo."""
        node, path_back = self, []
        while node:
            path_back.append(node)
            node = node.parent
        return list(reversed(path_back))

    def __lt__(self, node):
        return self.state < node.state

    def __eq__(self, other):
        """Este metodo se ejecuta cuando se compara nodos. Devuelve True cuando los estados son iguales"""
        return isinstance(other, Node) and self.state == other.state

    def __repr__(self):
        return f"<Node {self.state}>"

    def __hash__(self):
        return hash(self.state)

# Frontera tipo cola FIFO (first-in first-out) para BFS
class FIFOQueue(deque):
    """Una cola First-In-First-Out"""
    def pop(self):
        return self.popleft()


class FrontierPQ:
    """Una Frontera ordenada por una funcion de costo (Priority Queue)"""

    def __init__(self, initial, costfn=lambda node: node.path_cost):
        """Inicializa la Frontera con un nodo inicial y una funcion de costo especificada"""
        self.heap = []
        self.states = {}
        self.costfn = costfn
        self.add(initial)

    def add(self, node):
        """Agrega un nodo a la frontera."""
        cost = self.costfn(node)
        heapq.heappush(self.heap, (cost, node))
        self.states[node.state] = node

    def pop(self):
        """Remueve y retorna el nodo con minimo costo."""
        (cost, node) = heapq.heappop(self.heap)
        self.states.pop(node.state, None)  # Remove state
        return node

    def replace(self, node):
        """node reemplaza al nodo de la Frontera que tiene el mismo estado que node."""
        if node.state not in self:
            raise ValueError(f'{node.state} no tiene nada que reemplazar')
        for (i, (cost, old_node)) in enumerate(self.heap):
            if old_node.state == node.state:
                self.heap[i] = (self.costfn(node), node)
                heapq._siftdown(self.heap, 0, i)
                return

    def __contains__(self, state): return state in self.states
    def __len__(self): return len(self.heap)

In [104]:
#@title Visualización - No modifique este código, solo ejecute la celda

# Objetos en el espacio con paleta de colores mejorada
SPACE_OBJECTS = {
    'S': {'cost': float('inf'), 'value': 0,    'name': 'Sol',              'color': '#FFD700',    'size': 50},
    'E': {'cost': 1,            'value': 60,  'name': 'Objetivo',  'color': '#9400D3',    'size': 35},
    'P': {'cost': -0.5,            'value': 20,   'name': 'Planeta',          'color': '#00CED1',    'size': 20},
    'T': {'cost': -0.5,            'value': 40,   'name': 'Satélite',         'color': '#FF6347',    'size': 10},
    'A': {'cost': 5,            'value': -1,    'name': 'Asteroide',        'color': '#708090',    'size': 6},
    ' ': {'cost': 1,            'value': 0,    'name': 'Espacio',          'color': '#191970',    'size': 2}
}

# Diccionarios de acceso rápido para compatibilidad
COST_BLOCKS = {k: v['cost'] for k, v in SPACE_OBJECTS.items()}
VALUE_BLOCKS = {k: v['value'] for k, v in SPACE_OBJECTS.items()}
BLOCK_DEADLY = {'S'}
BLOCK_GOAL = {'E'}

SPACE_CONFIG = {
    'SUN': {'symbol': 'S', 'size': 50, 'color': '#FFD700'},
    'EYE': {'symbol': 'E', 'size': 35, 'color': '#9400D3'},
    'PLANET': {'symbol': 'P', 'size_range': (15, 25), 'color': '#00CED1'},
    'SATELLITE': {'symbol': 'T', 'size_range': (8, 12), 'color': '#FF6347'},
    'ASTEROID': {'symbol': 'A', 'size_range': (4, 7), 'color': '#708090'}
}

def get_space_object_property(block, property_name, default_value=None):
    """Obtiene una propiedad específica de un objeto espacial"""
    return SPACE_OBJECTS.get(block, {}).get(property_name, default_value)

def get_category(block):
    """Determina la categoría de un bloque basado en SPACE_CONFIG"""
    symbol_to_category = {
        'S': 'SUN', 'E': 'EYE', 'P': 'PLANET', 'T': 'SATELLITE', 'A': 'ASTEROID'
    }
    return symbol_to_category.get(block, None)

def get_category_color(block, category, pos, start_pos, goal_pos, object_colors):
    """Determina el color de un objeto basado en su categoría y posición """
    # Colores especiales para posiciones clave
    if pos == start_pos:
        return '#32CD32'  # Verde lima brillante para inicio
    elif pos == goal_pos:
        return '#9400D3'  # Violeta profundo para objetivo

    # Si el objeto tiene un color personalizado, usarlo
    if pos in object_colors:
        return object_colors[pos]

    # Colores cósmicos por categoría
    color_map = {
        'SUN': '#FFD700',      # Oro para el sol
        'EYE': '#9400D3',      # Violeta para el ojo
        'PLANET': '#00CED1',   # Turquesa para planetas
        'SATELLITE': '#FF6347', # Tomate para satélites
        'ASTEROID': '#708090'   # Gris pizarra para asteroides
    }
    return color_map.get(category, '#FFFFFF')

def simulate_system_3d(space, path_nodes):
    """
    Simula el sistema solar
    """

    # Obtener dimensiones del espacio
    x_max, y_max, z_max = space.getX(), space.getY(), space.getZ()
    start_pos = space.getStartCell()
    goal_pos = space.getGoalCell()

    # Obtener colores y tamaños de objetos si existen
    object_colors = getattr(space, 'object_colors', {})
    object_sizes = getattr(space, 'object_sizes', {})

    # Crear datos para objetos celestiales
    celestial_bodies = []
    for x in range(x_max):
        for y in range(y_max):
            for z in range(z_max):
                block = space.getBlock(x, y, z)
                cat = get_category(block)
                if cat is not None:
                    pos = (x, y, z)
                    size = object_sizes.get(pos, get_space_object_property(block, 'size', 6))
                    color = get_category_color(block, cat, pos, start_pos, goal_pos, object_colors)

                    # Get properties for hover text
                    cost = get_space_object_property(block, 'cost', 1)
                    value = get_space_object_property(block, 'value', 0)
                    name = get_space_object_property(block, 'name', 'Objeto Espacial')


                    celestial_bodies.append({
                        'x': z, 'y': y, 'z': x, 'category': cat, 'color': color,
                        'size': size, 'name': f"{name}<br>Costo: {cost}<br>Ganancia: {value}"
                    })

    # Agrupar objetos por categoría
    categories = {}
    for body in celestial_bodies:
        cat = body['category']
        if cat not in categories:
            categories[cat] = {'x': [], 'y': [], 'z': [], 'colors': [], 'sizes': [], 'names': []}
        for key in ['x', 'y', 'z', 'colors', 'sizes', 'names']:
            # Fix the property mapping
            if key == 'colors':
                prop = 'color'
            elif key == 'names':
                prop = 'name'
            elif key == 'sizes':
                prop = 'size'
            else:
                prop = key
            categories[cat][key].append(body[prop])


    # Símbolos espaciales mejorados por categoría
    symbols = {
        'SUN': 'circle',
        'EYE': 'diamond',
        'PLANET': 'circle',
        'SATELLITE': 'square',
        'ASTEROID': 'cross'
    }

    # Crear figura
    fig = go.Figure()

    # Agregar objetos celestiales
    for cat, data in categories.items():
        # Skip categories with only one object to avoid cluttering legend
        if len(data['x']) > 1:
            # Configurar efectos especiales según la categoría
            line_width = 2 if cat == 'SUN' else 1
            line_color = 'rgba(255, 255, 255, 0.8)' if cat == 'SUN' else 'rgba(255, 255, 255, 0.2)'

            fig.add_trace(go.Scatter3d(
                x=data['x'], y=data['y'], z=data['z'],
                mode='markers',
                marker=dict(
                    color=data['colors'],
                    size=data['sizes'],
                    symbol=symbols.get(cat, 'circle'),
                    line=dict(width=line_width, color=line_color),
                    opacity=0.9 if cat == 'SUN' else 0.8
                ),
                text=data['names'],
                hovertemplate='%{text}<extra></extra>',
                name=f'{cat.title()}s',
                legendgroup=cat
            ))
        else:
            # Add single objects without legend entry but with special effects
            line_width = 3 if cat == 'SUN' else 2 if cat == 'EYE' else 1
            line_color = 'rgba(255, 215, 0, 1)' if cat == 'SUN' else 'rgba(148, 0, 211, 1)' if cat == 'EYE' else 'rgba(255, 255, 255, 0.4)'

            fig.add_trace(go.Scatter3d(
                x=data['x'], y=data['y'], z=data['z'],
                mode='markers',
                marker=dict(
                    color=data['colors'],
                    size=data['sizes'],
                    symbol=symbols.get(cat, 'circle'),
                    line=dict(width=line_width, color=line_color),
                    opacity=1.0 if cat in ['SUN', 'EYE'] else 0.8
                ),
                text=data['names'],
                hovertemplate='%{text}<extra></extra>',
                showlegend=False
            ))

    # Agregar nave espacial
    start_x, start_y, start_z = start_pos
    fig.add_trace(go.Scatter3d(
        x=[start_z], y=[start_y], z=[start_x],
        mode='markers',
        marker=dict(
            color='#00FF00',  # Verde neón
            size=18,
            symbol='diamond',
            line=dict(width=3, color='#FFFFFF'),
            opacity=0.9
        ),
        name='Nave Espacial',
        hovertemplate='Nave Espacial<br>Posición de Inicio<br><extra></extra>'
    ))


    goal_x, goal_y, goal_z = goal_pos

    # Agregar trayectoria
    if len(path_nodes) > 1:
        trail_coords = [(node.state[2], node.state[1], node.state[0]) for node in path_nodes]
        trail_x, trail_y, trail_z = zip(*trail_coords)
        trail_text = [
            f"Posición: ({node.state[0]}, {node.state[1]}, {node.state[2]})<br>"
            f"Objeto: {get_space_object_property(space.getBlock(*node.state), 'name', 'Desconocido')}<br>"
            f"Costo: {get_space_object_property(space.getBlock(*node.state), 'cost', 1)}<br>"
            f"Ganancia: {get_space_object_property(space.getBlock(*node.state), 'value', 0)}"
            for node in path_nodes
        ]

        fig.add_trace(go.Scatter3d(
            x=trail_x, y=trail_y, z=trail_z,
            mode='lines+markers',
            line=dict(color='#FF4500', width=8, dash='solid'),
            marker=dict(color='#FF6347', size=5, symbol='circle', opacity=0.7),
            name='Ruta de Navegación',
            hovertemplate='%{text}<extra></extra>',
            text=trail_text
        ))

        # Add animation frames for the path with improved effects
        frames = []
        for i in range(len(path_nodes)):
            frame_data = []

            # Add all static objects with consistent styling
            for cat, data in categories.items():
                line_width = 2 if cat == 'SUN' else 1
                line_color = 'rgba(255, 255, 255, 0.8)' if cat == 'SUN' else 'rgba(255, 255, 255, 0.4)'

                if len(data['x']) > 1:
                    frame_data.append(go.Scatter3d(
                        x=data['x'], y=data['y'], z=data['z'],
                        mode='markers',
                        marker=dict(
                            color=data['colors'],
                            size=data['sizes'],
                            symbol=symbols.get(cat, 'circle'),
                            line=dict(width=line_width, color=line_color),
                            opacity=0.9 if cat == 'SUN' else 0.8
                        ),
                        text=data['names'],
                        hovertemplate='%{text}<extra></extra>',
                        name=f'{cat.title()}s',
                        legendgroup=cat
                    ))
                else:
                    line_width = 3 if cat == 'SUN' else 2 if cat == 'EYE' else 1
                    line_color = 'rgba(255, 215, 0, 1)' if cat == 'SUN' else 'rgba(148, 0, 211, 1)' if cat == 'EYE' else 'rgba(255, 255, 255, 0.4)'

                    frame_data.append(go.Scatter3d(
                        x=data['x'], y=data['y'], z=data['z'],
                        mode='markers',
                        marker=dict(
                            color=data['colors'],
                            size=data['sizes'],
                            symbol=symbols.get(cat, 'circle'),
                            line=dict(width=line_width, color=line_color),
                            opacity=1.0 if cat in ['SUN', 'EYE'] else 0.8
                        ),
                        text=data['names'],
                        hovertemplate='%{text}<extra></extra>',
                        showlegend=False
                    ))

            # Add ship at current position
            current_pos = path_nodes[i].state
            frame_data.append(go.Scatter3d(
                x=[current_pos[2]], y=[current_pos[1]], z=[current_pos[0]],
                mode='markers',
                marker=dict(
                    color='#00FF00',
                    size=18,
                    symbol='diamond',
                    line=dict(width=3, color='#FFFFFF'),
                    opacity=0.9
                ),
                name='Nave Espacial',
                hovertemplate='Nave Espacial<br>Navegando...<extra></extra>'
            ))

            # Add path up to current point with energy trail
            if i > 0:
                path_coords = [(path_nodes[j].state[2], path_nodes[j].state[1], path_nodes[j].state[0])
                              for j in range(i+1)]
                path_x, path_y, path_z = zip(*path_coords)
                frame_data.append(go.Scatter3d(
                    x=path_x, y=path_y, z=path_z,
                    mode='lines+markers',
                    line=dict(color="#FF9C77", width=5),
                    marker=dict(color="#DD7562", size=5, symbol='circle', opacity=0.7),
                    name='Ruta de Navegación',
                    hovertemplate='Paso %{pointIndex}<extra></extra>'
                ))

            frames.append(go.Frame(data=frame_data, name=f'frame{i}'))

        fig.frames = frames

    # Layout
    fig.update_layout(
        title=dict(
            text="Sistema solar",
            font=dict(size=20, color='#E6E6FA', family='Arial Black'),
            x=0.5
        ),
        scene=dict(
            xaxis_title="Dimensión Z",
            yaxis_title="Dimensión Y",
            zaxis_title="Dimensión X",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.5)),
            aspectmode='cube',
            bgcolor='rgb(8, 12, 30)',  # Azul
            xaxis=dict(
                backgroundcolor="rgb(8, 12, 30)",
                gridcolor="rgba(100, 149, 237, 0.3)",
                showbackground=True,
                zerolinecolor="rgba(100, 149, 237, 0.5)",
                title=dict(font=dict(color='#E6E6FA', size=14)),
                tickfont=dict(color='#E6E6FA')
            ),
            yaxis=dict(
                backgroundcolor="rgb(8, 12, 30)",
                gridcolor="rgba(100, 149, 237, 0.3)",
                showbackground=True,
                zerolinecolor="rgba(100, 149, 237, 0.5)",
                title=dict(font=dict(color='#E6E6FA', size=14)),
                tickfont=dict(color='#E6E6FA')
            ),
            zaxis=dict(
                backgroundcolor="rgb(8, 12, 30)",
                gridcolor="rgba(100, 149, 237, 0.3)",
                showbackground=True,
                zerolinecolor="rgba(100, 149, 237, 0.5)",
                title=dict(font=dict(color='#E6E6FA', size=14)),
                tickfont=dict(color='#E6E6FA')
            )
        ),
        paper_bgcolor='rgb(5, 8, 20)',  # Fondo negro espacial
        plot_bgcolor='rgb(5, 8, 20)',
        font=dict(color='#E6E6FA', family="Arial"),
        width=1300, height=950,
        legend=dict(
            x=0.02, y=0.98,
            bgcolor="rgba(8, 12, 30, 0.9)",
            bordercolor="#4169E1",
            borderwidth=2,
            font=dict(color='#E6E6FA', size=11)
        ),
        margin=dict(l=10, r=10, t=60, b=10),
        # Controles de animación mejorados
        updatemenus=[{
            'type': 'buttons',
            'showactive': False,
            'y': 0.02,
            'x': 0.02,
            'xanchor': 'left',
            'yanchor': 'bottom',
            'bgcolor': 'rgba(8, 12, 30, 0.9)',
            'bordercolor': '#4169E1',
            'borderwidth': 2,
            'font': {'color': '#E6E6FA'},
            'buttons': [
                {
                    'label': 'INICIAR NAVEGACIÓN',
                    'method': 'animate',
                    'args': [None, {
                        'frame': {'duration': 600, 'redraw': True},
                        'fromcurrent': True,
                        'transition': {'duration': 250}
                    }]
                },
                {
                    'label': 'SUSPENDER',
                    'method': 'animate',
                    'args': [[None], {
                        'frame': {'duration': 0, 'redraw': False},
                        'mode': 'immediate',
                        'transition': {'duration': 0}
                    }]
                },
                {
                    'label': 'REINICIAR',
                    'method': 'animate',
                    'args': [['frame0'], {
                        'frame': {'duration': 0, 'redraw': True},
                        'mode': 'immediate',
                        'transition': {'duration': 0}
                    }]
                }
            ]
        }]
    )

    fig.show()
    return fig

## Clase SpaceWorld
Implementa el mundo SpaceWorld


In [105]:
#No modificar
class SpaceWorld:
    def __init__(self, n=(25, 25, 25)):
        self.dimX, self.dimY, self.dimZ = n
        self.object_colors = {}
        self.object_sizes = {}
        self.grid = self.generate_solar_system(n)
        self.start = self.find_planet_position('P1')
        self.grid[self.start[0]][self.start[1]][self.start[2]] = 'P'
        self.goal = self.find_planet_position('E')

    def find_planet_position(self, symbol):
        for i in range(self.dimX):
            for j in range(self.dimY):
                for k in range(self.dimZ):
                    if self.grid[i][j][k] == symbol:
                        return (i, j, k)
        return None

    def isPassable(self, x, y, z):
        return self.grid[x][y][z] not in BLOCK_DEADLY

    def isBlocked(self, x, y, z):
        return not self.isPassable(x, y, z)

    def getBlock(self, x, y, z):
        return self.grid[x][y][z]

    def getX(self): return self.dimX
    def getY(self): return self.dimY
    def getZ(self): return self.dimZ
    def getStartCell(self): return self.start
    def getGoalCell(self): return self.goal

    def generate_solar_system(self, n, seed=42):
        """Función que genera el sistema solar en base a una semilla. No modificar"""
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        x, y, z = n
        space = np.full((x, y, z), ' ', dtype=object)
        center_x, center_y, center_z = x//2, y//2, z//2


        space[center_x, center_y, center_z] = 'S'
        self.object_sizes[(center_x, center_y, center_z)] = SPACE_CONFIG['SUN']['size']

        eye_x, eye_y, eye_z = 0, 0, 0
        space[eye_x, eye_y, eye_z] = 'E'
        self.object_sizes[(eye_x, eye_y, eye_z)] = SPACE_CONFIG['EYE']['size']

        start_x, start_y, start_z = x-1, y-1, z-1
        space[start_x, start_y, start_z] = 'P1'
        self.object_sizes[(start_x, start_y, start_z)] = random.randint(*SPACE_CONFIG['PLANET']['size_range'])

        num_planets = random.randint(35, 45)
        planets_placed = []
        for _ in range(num_planets):
            attempts = 0
            while attempts < 50:
                px = random.randint(2, x-3)
                py = random.randint(2, y-3)
                pz = random.randint(2, z-3)

                dist_from_sun = math.sqrt((px-center_x)**2 + (py-center_y)**2 + (pz-center_z)**2)
                dist_from_eye = math.sqrt((px-eye_x)**2 + (py-eye_y)**2 + (pz-eye_z)**2)

                if dist_from_sun > 3 and dist_from_eye > 1 and space[px, py, pz] == ' ':
                    space[px, py, pz] = 'P'
                    size = random.randint(*SPACE_CONFIG['PLANET']['size_range'])
                    self.object_sizes[(px, py, pz)] = size
                    planets_placed.append((px, py, pz))
                    break
                attempts += 1

        num_satellites = random.randint(14, 16)
        for _ in range(num_satellites):
            if planets_placed:
                parent_planet = random.choice(planets_placed)
                px, py, pz = parent_planet
                for dx in [-1, 0, 1]:
                    for dy in [-1, 0, 1]:
                        for dz in [-1, 0, 1]:
                            if dx == 0 and dy == 0 and dz == 0:
                                continue
                            sx, sy, sz = px + dx, py + dy, pz + dz
                            if (0 <= sx < x and 0 <= sy < y and 0 <= sz < z and
                                space[sx, sy, sz] == ' '):
                                space[sx, sy, sz] = 'T'
                                size = random.randint(*SPACE_CONFIG['SATELLITE']['size_range'])
                                self.object_sizes[(sx, sy, sz)] = size
                                break
                        else:
                            continue
                        break
                    else:
                        continue
                    break

        num_asteroids = random.randint(120, 160)
        for _ in range(num_asteroids):
            attempts = 0
            while attempts < 30:
                ax = random.randint(0, x-1)
                ay = random.randint(0, y-1)
                az = random.randint(0, z-1)

                if space[ax, ay, az] == ' ':
                    space[ax, ay, az] = 'A'
                    size = random.randint(*SPACE_CONFIG['ASTEROID']['size_range'])
                    self.object_sizes[(ax, ay, az)] = size
                    break
                attempts += 1

        return space

    def __str__(self):
        return f"SpaceWorld({self.dimX}x{self.dimY}x{self.dimZ}) - Objetivo en {self.goal}"

In [106]:
#Complete donde se indique COMPLETAR
class ExplorationProblem(SearchProblem):
    def __init__(self, space):
        """Constructor que recibe el espacio"""
        self.space = space
        self.initial = space.getStartCell()
        self.goal = space.getGoalCell()
        self.numNodesExpanded = 0
        self.expandedNodeSet = {}

    def __isValidState(self, state):
        """Retorna True si el estado es una celda válida y no mortal"""
        x, y, z = state

        #########################################################################################################
        # COMPLETAR - Verificar que todas coordenadas estén dentro de los límites del espacio
        if x < 0 or x >= self.space.getX():
            return False
        if y < 0 or y >= self.space.getY():
            return False
        if z < 0 or z >= self.space.getZ():
            return False
        #########################################################################################################
        return not self.space.isBlocked(x, y, z)

    def actions(self, state):
        """Retorna acciones legales: movimiento en 6 direcciones (3D)"""
        x, y, z = state
        actions = []

        directions = [
            (1, 0, 0, 'D'),   # Abajo/X+
            (-1, 0, 0, 'U'),  # Arriba/X-
            (0, 1, 0, 'R'),   # Derecha/Y+
            (0, -1, 0, 'L'),  # Izquierda/Y-
            (0, 0, 1, 'F'),   # Adelante/Z+
            (0, 0, -1, 'B')   # Atrás/Z-
        ]
        #########################################################################################################
        # COMPLETAR - Agregar a actions las direcciones válidas (de las 6 posibles). Tip: Usar self.__isValidState()
        for dx,dy,dz,accion in directions:
          print(dx , dy , dz , accion);
          new_state = (x + dx, y + dy, z + dz)
          print(new_state);
          if(self.__isValidState(new_state)):
            actions.append(accion)
        print(actions)
        #########################################################################################################
        return actions

    def result(self, state, action):
        """Retorna el estado resultante de ejecutar la acción"""
        x, y, z = state

        #########################################################################################################
        # COMPLETAR - Calcular las nuevas coordenadas según la acción
        dx, dy, dz = 0, 0, 0

        # Mapear acción a cambio de coordenadas
        if action == 'D': dx = 1
        if action == 'U': dx = -1
        if action == 'R': dy = 1
        if action == 'L': dy = -1
        if action == 'F': dz = 1
        if action == 'B': dz = -1

        new_state = (x + dx, y + dy, z + dz)
        #########################################################################################################

        return new_state

    def goal_test(self, state):
        """Retorna True si el estado es el objetivo"""
        return state == self.goal

    def path_cost(self, c, state1, action, state2):
        """Retorna el costo del camino, solo considera consumo de combustible (costo)"""
        x, y, z = state2
        block_type = self.space.getBlock(x, y, z)
        c += COST_BLOCKS.get(block_type, 1)
        return c

## Algoritmos de búsqueda
Se implementan algoritmos clásicos de búsqueda informada y no informada:

- **graph_search:** Algoritmo general de búsqueda con memoria de estados visitados. La frontera puede ser una cola FIFO (BFS) o una pila (DFS).
- **best_first_graph_search:** Expande el nodo con menor valor de una función de evaluación `f` (por ejemplo, heurística).
- **astar_search:** Caso especial de búsqueda best-first donde `f = path_cost + heuristic`, es decir, búsqueda A*.

Estos algoritmos permiten encontrar caminos óptimos en el entorno tridimensional.

In [107]:
# NO EDITAR ESTA CELDA - Algoritmos de búsqueda
def best_first_graph_search(problem, f):
    """Busca el objetivo expandiendo el nodo de la frontera con el menor valor de la funcion f.
    Memoriza estados visitados"""

    frontier = FrontierPQ(Node(problem.initial), f)  # frontera tipo cola de prioridad ordenada por f
    explored = set()     # memoria de estados visitados
    visited_nodes = []   # almacena nodos visitados durante la busqueda
    while frontier:
        node = frontier.pop()
        visited_nodes.append(node)
        if problem.goal_test(node.state):
            return node, visited_nodes
        explored.add(node.state)
        for action in problem.actions(node.state):
            child = node.child_node(problem, action)
            if child.state not in explored and child.state not in frontier:
                frontier.add(child)
            elif child.state in frontier:
                incumbent = frontier.states[child.state]
                if f(child) < f(incumbent):
                    frontier.replace(child)
    return None, visited_nodes

def graph_search(problem, frontier):
    """Algoritmo general de busqueda con memoria de estados visitados"""
    frontier.append(Node(problem.initial))
    explored = set()     # memoria de estados visitados
    visited_nodes = []   # almacena nodos visitados durante la busqueda
    while frontier:
        node = frontier.pop()
        visited_nodes.append(node)
        if problem.goal_test(node.state):
            return node, visited_nodes
        explored.add(node.state)

        frontier.extend(child for child in node.expand(problem)
                        if child.state not in explored and
                        child not in frontier)
    return None

def astar_search(problem, heuristic):
    """Algoritmo A* - caso especial de best_first_graph_search con f = path_cost + heuristic"""
    f = lambda node: node.path_cost + heuristic(node, problem)
    return best_first_graph_search(problem, f)

## Heurísticas espaciales para A*
Las heurísticas implementadas a continuación permiten estimar la distancia restante al objetivo para guiar el algoritmo A*:

- **Distancia Euclidiana:** Calcula la distancia en línea recta (como vuela una nave) entre el estado actual y el objetivo.
  
  $$
  h_{euclidiana}(n) = \sqrt{(x_{goal} - x_n)^2 + (y_{goal} - y_n)^2 + (z_{goal} - z_n)^2}
  $$

- **Distancia Manhattan:** Suma las diferencias absolutas en cada dimensión (movimiento ortogonal en 3D).

  $$
  h_{manhattan}(n) = |x_{goal} - x_n| + |y_{goal} - y_n| + |z_{goal} - z_n|
  $$

- **Heurística nula:** Siempre retorna 0, haciendo que A* se comporte como búsqueda de costo uniforme.

  $$
  h_{nula}(n) = 0
  $$

Estas heurísticas ayudan a priorizar los nodos más prometedores durante la búsqueda.

In [108]:
# DEBE COMPLETAR EL CÓDIGO DE CADA HEURÍSTICA

def euclidean_dist(node, problem):
    """Distancia en línea recta desde la posición actual hasta el objetivo"""
    x1, y1, z1 = node.state
    #########################################################################################################
    # COMPLETAR - Calcular la distancia euclidiana entre el estado actual y el objetivo
    #########################################################################################################
    return math.sqrt( math.pow(problem.goal[0] - x1, 2) + math.pow(problem.goal[1] - y1, 2) + math.pow(problem.goal[2] - z1, 2) )

def manhattan_dist(node, problem):
    """Distancia Manhattan (suma de diferencias absolutas)"""
    x1, y1, z1 = node.state
    #########################################################################################################
    # COMPLETAR - Calcular la distancia Manhattan entre el estado actual y el objetivo
    return abs( problem.goal[0] - x1)  + abs(problem.goal[1] - y1) + abs(problem.goal[2] - z1)

    #########################################################################################################

def null_heuristic(node, problem):
    """Heurística nula (A* se convierte en búsqueda de costo uniforme)"""
    #########################################################################################################
    # COMPLETAR - Implementar la heurística nula
    return 0
    #########################################################################################################



In [109]:
mundo = SpaceWorld((15, 15, 15))
problem = ExplorationProblem(mundo)

heuristics = [
    ("Euclidiana", euclidean_dist),
    ("Manhattan", manhattan_dist),
    ("Nula", null_heuristic),
]

## Probar las heurísticas
Ejecute el algoritmo A* con cada una de las heurísticas implementadas

In [110]:
# No hace falta modificar nada aquí
for nombre, heur in heuristics:
    print(f"\nProbando heurística: {nombre}")
    nodo_solucion, nodos_visitados = astar_search(problem, heur)
    if nodo_solucion:
        ganancia = 0
        for node in nodo_solucion.path():
            block = mundo.getBlock(*node.state)
            ganancia += VALUE_BLOCKS.get(block, 0)
        print(f"Solución encontrada con costo {nodo_solucion.path_cost}")
        print(f"Ganancia acumulada: {ganancia}")
        print(f"Nodos expandidos: {len(nodos_visitados)}")
        print(f"Número de acciones: {len(nodo_solucion.solution())}")
        print("Ruta de acciones:", nodo_solucion.solution())
        simulate_system_3d(mundo, nodo_solucion.path())
    else:
        print("No se encontró solución.")


Probando heurística: Euclidiana
1 0 0 D
(15, 14, 14)
-1 0 0 U
(13, 14, 14)
0 1 0 R
(14, 15, 14)
0 -1 0 L
(14, 13, 14)
0 0 1 F
(14, 14, 15)
0 0 -1 B
(14, 14, 13)
['U', 'L', 'B']
1 0 0 D
(14, 14, 14)
-1 0 0 U
(12, 14, 14)
0 1 0 R
(13, 15, 14)
0 -1 0 L
(13, 13, 14)
0 0 1 F
(13, 14, 15)
0 0 -1 B
(13, 14, 13)
['D', 'U', 'L', 'B']
1 0 0 D
(15, 13, 14)
-1 0 0 U
(13, 13, 14)
0 1 0 R
(14, 14, 14)
0 -1 0 L
(14, 12, 14)
0 0 1 F
(14, 13, 15)
0 0 -1 B
(14, 13, 13)
['U', 'R', 'L', 'B']
1 0 0 D
(15, 14, 13)
-1 0 0 U
(13, 14, 13)
0 1 0 R
(14, 15, 13)
0 -1 0 L
(14, 13, 13)
0 0 1 F
(14, 14, 14)
0 0 -1 B
(14, 14, 12)
['U', 'L', 'F', 'B']
1 0 0 D
(14, 13, 14)
-1 0 0 U
(12, 13, 14)
0 1 0 R
(13, 14, 14)
0 -1 0 L
(13, 12, 14)
0 0 1 F
(13, 13, 15)
0 0 -1 B
(13, 13, 13)
['D', 'U', 'R', 'L', 'B']
1 0 0 D
(14, 14, 13)
-1 0 0 U
(12, 14, 13)
0 1 0 R
(13, 15, 13)
0 -1 0 L
(13, 13, 13)
0 0 1 F
(13, 14, 14)
0 0 -1 B
(13, 14, 12)
['D', 'U', 'L', 'F', 'B']
1 0 0 D
(15, 13, 13)
-1 0 0 U
(13, 13, 13)
0 1 0 R
(14, 14, 13


Probando heurística: Manhattan
1 0 0 D
(15, 14, 14)
-1 0 0 U
(13, 14, 14)
0 1 0 R
(14, 15, 14)
0 -1 0 L
(14, 13, 14)
0 0 1 F
(14, 14, 15)
0 0 -1 B
(14, 14, 13)
['U', 'L', 'B']
1 0 0 D
(14, 14, 14)
-1 0 0 U
(12, 14, 14)
0 1 0 R
(13, 15, 14)
0 -1 0 L
(13, 13, 14)
0 0 1 F
(13, 14, 15)
0 0 -1 B
(13, 14, 13)
['D', 'U', 'L', 'B']
1 0 0 D
(13, 14, 14)
-1 0 0 U
(11, 14, 14)
0 1 0 R
(12, 15, 14)
0 -1 0 L
(12, 13, 14)
0 0 1 F
(12, 14, 15)
0 0 -1 B
(12, 14, 13)
['D', 'U', 'L', 'B']
1 0 0 D
(13, 13, 14)
-1 0 0 U
(11, 13, 14)
0 1 0 R
(12, 14, 14)
0 -1 0 L
(12, 12, 14)
0 0 1 F
(12, 13, 15)
0 0 -1 B
(12, 13, 13)
['D', 'U', 'R', 'L', 'B']
1 0 0 D
(12, 13, 14)
-1 0 0 U
(10, 13, 14)
0 1 0 R
(11, 14, 14)
0 -1 0 L
(11, 12, 14)
0 0 1 F
(11, 13, 15)
0 0 -1 B
(11, 13, 13)
['D', 'U', 'R', 'L', 'B']
1 0 0 D
(11, 13, 14)
-1 0 0 U
(9, 13, 14)
0 1 0 R
(10, 14, 14)
0 -1 0 L
(10, 12, 14)
0 0 1 F
(10, 13, 15)
0 0 -1 B
(10, 13, 13)
['D', 'U', 'R', 'L', 'B']
1 0 0 D
(10, 13, 14)
-1 0 0 U
(8, 13, 14)
0 1 0 R
(9, 14, 1

Se truncaron las últimas líneas 5000 del resultado de transmisión.
0 0 -1 B
(1, 1, 10)
['D', 'U', 'R', 'L', 'F', 'B']
1 0 0 D
(2, 2, 1)
-1 0 0 U
(0, 2, 1)
0 1 0 R
(1, 3, 1)
0 -1 0 L
(1, 1, 1)
0 0 1 F
(1, 2, 2)
0 0 -1 B
(1, 2, 0)
['D', 'U', 'R', 'L', 'F', 'B']
1 0 0 D
(2, 3, 0)
-1 0 0 U
(0, 3, 0)
0 1 0 R
(1, 4, 0)
0 -1 0 L
(1, 2, 0)
0 0 1 F
(1, 3, 1)
0 0 -1 B
(1, 3, -1)
['D', 'U', 'R', 'L', 'F']
1 0 0 D
(2, 4, 13)
-1 0 0 U
(0, 4, 13)
0 1 0 R
(1, 5, 13)
0 -1 0 L
(1, 3, 13)
0 0 1 F
(1, 4, 14)
0 0 -1 B
(1, 4, 12)
['D', 'U', 'R', 'L', 'F', 'B']
1 0 0 D
(2, 4, 14)
-1 0 0 U
(0, 4, 14)
0 1 0 R
(1, 5, 14)
0 -1 0 L
(1, 3, 14)
0 0 1 F
(1, 4, 15)
0 0 -1 B
(1, 4, 13)
['D', 'U', 'R', 'L', 'B']
1 0 0 D
(2, 7, 0)
-1 0 0 U
(0, 7, 0)
0 1 0 R
(1, 8, 0)
0 -1 0 L
(1, 6, 0)
0 0 1 F
(1, 7, 1)
0 0 -1 B
(1, 7, -1)
['D', 'U', 'R', 'L', 'F']
1 0 0 D
(2, 8, 1)
-1 0 0 U
(0, 8, 1)
0 1 0 R
(1, 9, 1)
0 -1 0 L
(1, 7, 1)
0 0 1 F
(1, 8, 2)
0 0 -1 B
(1, 8, 0)
['D', 'U', 'R', 'L', 'F', 'B']
1 0 0 D
(2, 9, 6)
-1 0 0 U
(0, 

# Preguntas




0) Completar el código faltante. (6 ptos)
1) En cuanto a las soluciones encontradas por A*, ¿ellas serán siempre óptimas?.
  ¿Qué requisitos necesitan la heurística para garantizar soluciones óptimas?.
  ¿Estos requisitos se cumplen en todas las heurísticas planteadas para el laboratorio?. Explique. (3 puntos)




Respuesta:
No siempre, depende de la heuristica. Para encontrar la solución optima se requiere un valor ideal que varia dependiendo de cada una, y este requisito no se cumple para la última pues es nula.


2) Determine el nivel de dominancia entre las heurísticas. Explique y justifique. Mencione el orden de dominancia. (2 puntos)




Respuesta:

La mas dominante es la Euclidiana pues es la heuristica con la base mas cercana al costo ideal (linea recta). Luego le seguiria la Manhattan pues asume movimiento ortogonal y por ultimo la nula.


3) ¿En cuanto al costo temporal y en memoria, con cuál heurística se visita menos nodos? ¿Por qué? Justifique en base a la teoría y a lo experimentado. (3 puntos)



Respuesta:
Con la Euclidiana, se tiene un camino recto entre el nodo actual y el objetivo. Manhattan prueba caminos mas variados dado a que su heuristica no es mas directa sino algo mas "aterrizada"



4) Explique cómo se podría adaptar un algoritmo de búsqueda local (como Hill Climbing o Simulated Annealing) al problema de encontrar elementos en el espacio que tengan el máximo valor. Describa la representación de los estados, la función objetivo, la definición de las acciones y vecindad, y las estrategias para evitar quedar atrapado en óptimos locales (2 puntos)



Respuesta:
Aplicando Simulated Annealing podriamos comenzar tomando puntos aleatorios en el mapa y analizar sus vecinos e ir saltando comparando el valor de cada ruta para decidir con que punto continuar, empleando un algoritmo de temperatura que gracias a su drop rate poco a poco nos acerca a la solución optima global
Representación de estados: mediante ruta, pues es como el "tablero", va cambiando conforme demos un paso adelante en el algoritmo y con ella sus vecinos
Función Objetivo: minimizar la distancia entre el punto de partida y destino
Definición de acciones: el set 6 de movimientos  
Vencindad: rutas generadas a partir del punto actual
Las estrategias serian mediante el uso del valor de temperatura para obtener la completitud que proporciona el Simulated Annealing, de esta forma se evitaria caer en un maximo local


5) ¿Cómo se controla el grado de exploración (encontrar nuevas soluciones) en un algoritmo Simulated Annealing?. Relacionelo en cuanto a la convergencia y a la optimalidad del algoritmo  ¿Cuáles son las ventajas de Simulating Annealing respecto de Hill Climbing?(2 ptos)



Respuesta:

Por último, las ventajas de este algoritmo respecto al Hill Climbing son que hereda la capacidad de completitud del algoritmo randonomico, manteniendo consigo la eficiencia del primero. Otra ventaja importante es el control mediante la implementación de la logica de temperatura y la probabilidad con respecto a la diferencia entre dos puntos lo que garantiza que conforme esta descienda es mas probable tener el máximo global.


6) En el problema del Exploration World  se muestra que los elementos del espacio, ademas de un costo, tienen un puntaje de 'valor'. ¿Cuál es el propósito de este 'valor'? ¿Se está optimizando actualmente este valor? Explique cómo podría modificarse el planteamiento del algoritmo A* para que este valor sea considerado en la busqueda y encuentre el camino que minimize el costo y maximize el valor de los elementos del camino. Proponga un caso real donde pueda ser aplicado (2 ptos)



Respuesta:
Su proposito es asignar prioridad a ciertos nodos o puntos debido a la información en este caso que puedan proporcionar a la investigación. Actualmente no se esta tomando en cuenta para el algoritmo, y si quisiesemos modificar su comportamiento para tenerlo en cuenta como parametro lo que se podria hacer es incluir su valor dentro de las formulas para calculo de distancia. Yo lo haria de la siguiente forma: tomo el valor de la distancia y lo divido entre una potencia de 10 y luego le sumo el valor del punto de interes. De esta manera es el valor el parametro prioritario en cuanto al calculo de distancias   
